> **Public (scrubbed) copy:** Real manufacturer / brand labels are replaced with **placeholders** (`MFG_ALPHA`, `MFG_BETA`, …, `FRANCHISE_A`). For this notebook to run on your CSV, those columns must use the **same placeholder strings**, or change the literals in the code. **No dataset is included.** Outputs are cleared.


# Predictive analytics: plant-based / manufacturer retail panel

**Context:** Masters coursework (predictive modeling) using retail-style panel data: product attributes, distribution (ACV), pricing, and unit sales.

**What this notebook does**
1. **Feature engineering** — Collapse high-cardinality text fields (form, flavor, package, product type) into business-facing groups; derive velocity and promotional metrics.
2. **LASSO regression** — Predict unit sales with one-hot encoded categories + numeric features; inspect sparse coefficients and holdout R².
3. **Monthly models** — Refit LASSO by calendar month to see stability of signals.
4. **Appendices** — Single MFG (MFG_ALPHA) sklearn pipeline; OLS with manufacturer interactions, scenario plots, and VIF.

**Data:** The underlying extract is **not** included (employer / policy). Clone the repo, create `data/`, and place your own `filtered_manufacturers_data.csv` with the columns used below, or adapt `DATA_PATH`.


In [ ]:
from pathlib import Path
import os

import pandas as pd

# Optional override:
# export PREDICTIVE_ROOT="/Users/rukmangadaruku/Library/CloudStorage/OneDrive-TheUniversityofTexasatDallas/Sem 2/proj for git upload"
_MANUAL_PREDICTIVE_ROOT = Path("/Users/rukmangadaruku/Library/CloudStorage/OneDrive-TheUniversityofTexasatDallas/Sem 2/proj for git upload")


def find_project_root() -> Path:
    """Resolve folder that contains this notebook and the filtered CSV."""
    if _MANUAL_PREDICTIVE_ROOT is not None:
        return Path(_MANUAL_PREDICTIVE_ROOT).expanduser().resolve()
    env = os.environ.get("PREDICTIVE_ROOT", "").strip()
    if env:
        return Path(env).expanduser().resolve()

    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "filtered_manufacturers_data.csv").is_file():
            return p
        if (p / "data" / "filtered_manufacturers_data.csv").is_file():
            return p
        if (p / "Data" / "filtered_manufacturers_data.csv").is_file():
            return p

    fallback_root = Path("/Users/rukmangadaruku/Library/CloudStorage/OneDrive-TheUniversityofTexasatDallas/Sem 2/proj for git upload")
    if fallback_root.is_dir():
        return fallback_root
    return here


def resolve_data_csv(base: Path) -> Path:
    candidates = [
        base / "filtered_manufacturers_data.csv",
        base / "data" / "filtered_manufacturers_data.csv",
        base / "Data" / "filtered_manufacturers_data.csv",
    ]
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError(
        "Could not find filtered_manufacturers_data.csv. Tried:
"
        + "
".join(f"  - {p}" for p in candidates)
    )


ROOT = find_project_root()
DATA_PATH = resolve_data_csv(ROOT)

df_raw = pd.read_csv(DATA_PATH)
data = df_raw.copy()
print(f"Using ROOT={ROOT}")
print(f"Loaded {len(data):,} rows from {DATA_PATH}")


## 1. Feature engineering

Raw `Form`, `Flavor / Scent`, `Package`, and `Product Type` fields are mapped to **grouped categories** to reduce sparsity. Derived fields:
- **Velocity** — unit sales per ACV weighted distribution.
- **Incremental Sales %** — incremental units relative to unit sales.

Small constants fill missing denominators to avoid divide-by-zero; adjust for production use if needed.


In [ ]:
def simplify_form(x):
    x = str(x).strip().upper()
    if any(term in x for term in ['BURGER', 'PATTY', 'SLIDER']):
        return 'BURGER/PATTY'
    elif any(term in x for term in ['NUGGET', 'TENDER', 'STRIP', 'FUN NUGGETS', 'POPCORN', 'FINGER', 'FRIES']):
        return 'NUGGET/TENDER/STRIP'
    elif any(term in x for term in ['GROUND', 'CRUMBLE', 'PULLED', 'SHREDDED', 'SHREDS']):
        return 'GROUND/CRUMBLE'
    elif any(term in x for term in ['SAUSAGE', 'BRATWURST', 'LINK', 'DINNER SAUSAGE', 'BREAKFAST SAUSAGE']):
        return 'SAUSAGE/LINK'
    elif any(term in x for term in ['HOT DOG', 'FRANK']):
        return 'HOT DOG'
    elif any(term in x for term in ['MEATBALL']):
        return 'MEATBALL'
    elif any(term in x for term in ['STEAK', 'FILET', 'CUTLET', 'ROAST']):
        return 'STEAK/CUT/ROAST'
    elif 'BLOCK' in x or 'CUBE' in x or 'LOAF' in x:
        return 'BLOCK/CUBE/LOAF'
    else:
        return 'OTHER'

data['Form Grouped'] = data['Form'].apply(simplify_form)

def simplify_flavor(x):
    x = str(x).strip().upper()
    if any(term in x for term in ['SPICY', 'CHIPOTLE', 'HOT', 'HABANERO', 'SRIRACHA']):
        return 'SPICY'
    elif any(term in x for term in ['BBQ', 'BARBEQUE', 'GRILLED', 'KOREAN BARBEQUE']):
        return 'BBQ'
    elif any(term in x for term in ['GARLIC', 'HERB', 'BASIL', 'OREGANO', 'ITALIAN']):
        return 'HERB/GARLIC'
    elif any(term in x for term in ['CURRY', 'MASALA', 'TANDOORI', 'INDIAN', 'BOMBAY', 'MADRAS']):
        return 'CURRY/EXOTIC'
    elif any(term in x for term in ['SMOKE', 'SMOKY', 'APPLEWOOD', 'HICKORY']):
        return 'SMOKY'
    elif 'SWEET' in x:
        return 'SWEET'
    elif any(term in x for term in ['PLAIN', 'ORIGINAL', 'REGULAR', 'CLASSIC', 'BASIC']):
        return 'PLAIN/ORIGINAL'
    else:
        return 'OTHER'

data['Flavor Grouped'] = data['Flavor / Scent'].apply(simplify_flavor)


def simplify_package(x):
    x = str(x).strip().upper()
    if 'BOX' in x or 'CARTON' in x:
        return 'BOXED'
    elif 'BAG' in x or 'POUCH' in x or 'PEG' in x:
        return 'BAGGED'
    elif 'VACUUM' in x:
        return 'VACUUM PACKED'
    elif 'WRAP' in x or 'WRAPPED' in x:
        return 'WRAPPED'
    elif 'TUB' in x or 'CONTAINER' in x:
        return 'TUB/CONTAINER'
    else:
        return 'OTHER'

data['Package Grouped'] = data['Package'].apply(simplify_package)

def simplify_product_type(x):
    x = str(x).strip().upper()
    if 'TOFU' in x:
        return 'TOFU'
    elif 'TEMPEH' in x:
        return 'TEMPEH'
    elif 'FALAFEL' in x:
        return 'FALAFEL'
    elif 'SEAFOOD' in x or 'FISH' in x:
        return 'SEAFOOD SUBSTITUTE'
    elif 'SOY' in x:
        return 'SOY'
    elif 'SEITAN' in x:
        return 'SEITAN'
    elif 'PLANT' in x or 'MEAT SUBSTITUTE' in x:
        return 'PLANT-BASED SUBSTITUTE'
    else:
        return 'OTHER'

data['Product Type Grouped'] = data['Product Type'].apply(simplify_product_type)


In [ ]:
# SAFETY: Fill NaN ACV Weighted Distribution to avoid divide-by-zero errors
data['ACV Weighted Distribution'] = data['ACV Weighted Distribution'].fillna(0.01)

# Create Velocity: Unit Sales per ACV Weighted Distribution
data['Velocity'] = data['Unit Sales'] / data['ACV Weighted Distribution']

# SAFETY: Fill NaN Unit Sales to avoid divide-by-zero
data['Unit Sales'] = data['Unit Sales'].fillna(0.01)

# Create Incremental Sales %: (Incremental Units / Unit Sales) * 100
data['Incremental Sales %'] = (data['Incremental Units'] / data['Unit Sales']) * 100


## 2. LASSO regression (main model)

**Target:** `Unit Sales`. **Features:** grouped categories plus `Price per Unit` and `ACV Weighted Distribution`. Categoricals are one-hot encoded (`drop='first'`). **LassoCV** picks the penalty; non-zero coefficients are interpreted as the active subset of predictors after regularization.


In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import numpy as np

target = data['Unit Sales']
features = data[['Form Grouped', 'Flavor Grouped', 'Package Grouped',
                 'Product Type Grouped', 'Price per Unit', 'ACV Weighted Distribution']]
categorical_cols = ['Form Grouped', 'Flavor Grouped', 'Package Grouped', 'Product Type Grouped']
numeric_cols = ['Price per Unit', 'ACV Weighted Distribution']
X_categorical = features[categorical_cols]
X_numeric = features[numeric_cols]
# OneHot encode
encoder = OneHotEncoder(drop='first', sparse_output=False)
X_cat_encoded = encoder.fit_transform(X_categorical)

X_all = np.hstack([X_cat_encoded, X_numeric.values])
X_train, X_test, y_train, y_test = train_test_split(X_all, target, test_size=0.2, random_state=42)

# Fit LASSO with cross-validation to choose alpha
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train, y_train)

print("\nBest alpha (regularization strength):", lasso.alpha_)

# Feature importance: non-zero coefficients
feature_names = encoder.get_feature_names_out(categorical_cols).tolist() + numeric_cols
coef = pd.Series(lasso.coef_, index=feature_names)

print("\nNon-zero feature coefficients (important predictors):")
print(coef[coef != 0].sort_values(ascending=False))

# Optional: Evaluate performance
print("\nTrain R^2 score:", lasso.score(X_train, y_train))
print("Test R^2 score:", lasso.score(X_test, y_test))


import matplotlib.pyplot as plt
import seaborn as sns

# Filter non-zero coefficients
non_zero_coef = coef[coef != 0].sort_values()

# Plot (DataFrame + hue avoids seaborn v0.14 deprecation)
plot_df = non_zero_coef.reset_index()
plot_df.columns = ['feature', 'coef']
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="coef", y="feature", hue="feature", palette="viridis", legend=False)
plt.title('LASSO Regression: Important Feature Coefficients')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


## 3. Stability across months

The same LASSO setup is fit **per calendar month** (from a `Time` column). Months with fewer than 30 rows are skipped. Use this to see whether key drivers are stable seasonally or data-sparse.


In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import OneHotEncoder
import numpy as np

data['Time'] = pd.to_datetime(data['Time'])
# Extract month (1 to 12)
data['Month'] = data['Time'].dt.month

# Prepare output storage
monthly_results = {}

# Define features
categorical_cols = ['Form Grouped', 'Flavor Grouped', 'Package Grouped', 'Product Type Grouped']
numeric_cols = ['Price per Unit', 'ACV Weighted Distribution']

# Loop through each month (1 to 12)
for month in sorted(data['Month'].unique()):
    print(f"\n=== Month: {month} ===")
    month_data = data[data['Month'] == month]

    # Skip if too few samples
    if len(month_data) < 30:
        print(f"Not enough data points for Month {month}, skipping.")
        continue

    target = month_data['Unit Sales']
    features = month_data[categorical_cols + numeric_cols]

    # OneHot encode
    encoder = OneHotEncoder(drop='first', sparse_output=False)
    X_cat_encoded = encoder.fit_transform(features[categorical_cols])
    X_all = np.hstack([X_cat_encoded, features[numeric_cols].values])

    # LASSO
    lasso = LassoCV(cv=5, random_state=42)
    lasso.fit(X_all, target)

    feature_names = encoder.get_feature_names_out(categorical_cols).tolist() + numeric_cols
    coef = pd.Series(lasso.coef_, index=feature_names)

    print("Alpha:", lasso.alpha_)
    print("Non-zero features:\n", coef[coef != 0].sort_values(ascending=False))

    # Store results
    monthly_results[month] = coef[coef != 0]


## Appendix A — Single-manufacturer subset (MFG_ALPHA)

Same sklearn `Pipeline` idea on rows where manufacturer name contains `MFG_ALPHA`: standardized numerics, one-hot categoricals, LassoCV. Useful as a sensitivity check versus the pooled grouped-feature model.


In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns

# Reuse `df_raw` from the first cell; filter to rows with Manufacturer Name matching MFG_ALPHA
df = df_raw.copy()
df_mfg_alpha = df[df['Manufacturer Name'].str.contains('MFG_ALPHA', case=False, na=False)]

# Define target & features
y = df_mfg_alpha['Unit Sales']
X = df_mfg_alpha[
    ['Price per Unit', 'ACV Weighted Distribution', 
     'Package', 'Form', 'Flavor / Scent', 'Product Type']
]

# Drop any rows with missing values in X or y (to avoid errors)
df_model = pd.concat([X, y], axis=1).dropna()
X = df_model.drop(columns='Unit Sales')
y = df_model['Unit Sales']

numeric_features = ['Price per Unit', 'ACV Weighted Distribution']
categorical_features = ['Package', 'Form', 'Flavor / Scent', 'Product Type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ]
)

# Build pipeline: preprocessing + LASSO with cross-validation
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lasso', LassoCV(cv=5, random_state=0))
])

# Fit the model
pipeline.fit(X, y)
ohe = pipeline.named_steps['preprocessor'].named_transformers_['cat']
encoded_cat_names = ohe.get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(encoded_cat_names)

lasso_coef = pipeline.named_steps['lasso'].coef_
# Create DataFrame for visualization
coef_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient': lasso_coef
})

# Only keep non-zero coefficients
coef_df = coef_df[coef_df['Coefficient'] != 0]

# Sort by absolute value and take top 15
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)
top_coef_df = coef_df.head(15)
# === Existing code assumed to be already executed above ===

# Print best alpha value chosen by LassoCV
print(f"Best alpha (regularization strength): {pipeline.named_steps['lasso'].alpha_:.4f}")

# Print R² score on the full dataset
r2_score = pipeline.score(X, y)
print(f"Model R² score on full dataset: {r2_score:.4f}")


# Filter non-zero coefficients
coef_df = coef_df[coef_df['Coefficient'] != 0]

# Sort by absolute value and get top 15
top_coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index).head(15)

# Print top 15 important features
print("Top 15 Important Features (by LASSO Coefficients):")
for i, row in top_coef_df.iterrows():
    print(f"{row['Feature']:40s} {row['Coefficient']: .4f}")

# Plot
plt.figure(figsize=(8,6))
sns.barplot(x='Coefficient', y='Feature', data=top_coef_df, hue='Feature', palette='viridis', legend=False)
plt.title('Top 15 Important Features (LASSO Regression for MFG_ALPHA)')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


## Appendix B — OLS with manufacturer interactions

**Linear model (statsmodels):** Unit sales on grouped form and flavor, manufacturer fixed effects, price discount %, ACV, and **manufacturer × price discount** interactions. Follow-on cells summarize coefficients visually (including hard-coded coefficient tables from a prior model fit), discount scenarios, ACV partial effect, and VIF.

Run cells in order; `df_selected` is built from `df` in the cells below.


In [ ]:
df = df_raw.copy()


In [ ]:
# Calculate price discount percentage (normalized)
df['Price_Discount_Pct'] = (df['Price per Unit No Merch'] - df['Price per Unit']) / df['Price per Unit No Merch']

# Handle potential division by zero or NaNs
df['Price_Discount_Pct'] = df['Price_Discount_Pct'].replace([float('inf'), -float('inf')], pd.NA)
df['Price_Discount_Pct'] = df['Price_Discount_Pct'].fillna(0)

# Define the selected columns
selected_columns = [
     'Price_Discount_Pct',
     'ACV Weighted Distribution',
     'Form','Flavor / Scent',
     'Manufacturer Name',
     'Unit Sales'  # Target
]

# Create the new DataFrame
df_selected = df[selected_columns].copy()

In [ ]:
# Final grouped form dictionary
form_group_mapping = {
    "burger": "patty", "burger patty": "patty", "patty": "patty", "breakfast patty": "patty",
    "sausage patty": "patty", "slider": "patty",

    "link": "sausage", "breakfast link": "sausage", "dinner link": "sausage",
    "dinner sausage link": "sausage", "breakfast sausage link": "sausage",
    "breakfast sausage patty": "sausage", "breakfast sausage roll": "sausage",
    "sausage": "sausage", "bratwurst": "sausage", "frank": "sausage", "hot dog": "sausage",
    "split rope": "sausage", "rope": "sausage", "roll": "sausage", "chub": "sausage",

    "nugget": "nugget", "fun nuggets": "nugget", "bite": "nugget", "breakfast bites": "nugget",
    "popcorn": "nugget", "popper": "nugget", "dipper": "nugget", "bings": "nugget",
    "bar": "nugget", "stick": "nugget", "fries": "nugget",

    "ground": "ground", "crumble": "ground", "shredded": "ground", "shreds": "ground", "pulled": "ground",

    "strip": "pieces", "tender": "pieces", "tenders": "pieces", "wing": "pieces",
    "finger": "pieces", "drumstick": "pieces", "piece": "pieces",
    "spare ribs": "pieces", "riblet": "pieces", "tip": "pieces",

    "sliced": "sliced", "slice": "sliced", "deli sliced": "sliced", "ultra thin slice": "sliced",
    "cut": "sliced", "cutlet": "sliced", "cube": "sliced", "chunk": "sliced",
    "diced": "sliced", "filet": "sliced", "steak": "sliced",

    "roast": "whole", "whole": "whole", "loaf": "whole", "meat loaf": "whole",
    "breast": "whole", "block": "whole",

    "meatball": "ball", "ball": "ball", "cake": "ball", "bao bun": "ball",

    "gizzard": "unknown", "not stated on package": "unknown"
}


df_selected["Form_Grouped"] = df_selected["Form"].str.lower().map(form_group_mapping).fillna("unknown")
df_selected = df_selected[df_selected["Form_Grouped"] != "unknown"]

# Save if needed
# df.to_csv("forms_grouped_output.csv", index=False)

In [ ]:
# Final grouped flavor dictionary
flavor_group_mapping = {
    # === Neutral / Original ===
    "regular": "original", "original": "original", "classic": "original",
    "traditional": "original", "simply seasoned": "original", "lightly seasoned": "original",
    "seasoned": "original", "original turkey": "original", "original sausage": "original",
    "chicken": "original", "roasted turkey": "original", "turkey roast": "original",
    "turkey": "original", "ham": "original", "smoked ham": "original", "sausage": "original",
    "breakfast sausage": "original", "italian sausage": "original", "bratwurst": "original",
    "hot dog": "original", "bacon": "original", "beef": "original", "meat lovers": "original",
    "philly steak": "original", "canadian bacon": "original", "corned beef": "original",

    # === BBQ / Smoky ===
    "barbeque": "bbq", "bbq": "bbq", "smoky": "bbq", "flame grilled": "bbq",
    "char grilled": "bbq", "grilled": "bbq", "hickory smoked": "bbq", "smoked": "bbq",
    "wood smoked": "bbq", "smoked tomato": "bbq", "grandpa mels barbeque": "bbq",
    "sweet barbeque": "bbq", "sweet barbeque chicken": "bbq", "smoky maple bacon": "bbq",
    "smoked apple sage": "bbq", "smoky & spicy": "bbq", "smoked salt & pepper steak": "bbq",
    "spanish smoked": "bbq",

    # === Spicy ===
    "spicy": "spicy", "buffalo": "spicy", "buffalo tempeh": "spicy", "spicy black bean": "spicy",
    "spicy chipotle black bean": "spicy", "chipotle": "spicy", "chipotle lime": "spicy",
    "mexican chipotle": "spicy", "hot italian": "spicy", "spicy italian": "spicy",
    "mama mia spicy italian": "spicy", "hot & spicy sausage": "spicy", "spicy sausage": "spicy",
    "spicy garlic": "spicy", "spicy thai": "spicy", "spicy falafel": "spicy",
    "zesty italian": "spicy", "zesty mexican": "spicy", "kickin": "spicy", "nashville hot": "spicy",
    "tandoori spice": "spicy", "spicy green chili": "spicy", "mild hot": "spicy",
    "spicy italian hempseed": "spicy", "korean": "spicy", "buffalo style cauliflower": "spicy",
    "curry": "spicy", "coconut curry": "spicy", "madras curry": "spicy",
    "indian spiced masala": "spicy", "masala": "spicy", "curried sweet potato": "spicy",
    "spicey indian vegetable": "spicy",

    # === Sweet ===
    "maple": "sweet", "maple sausage": "sweet", "apple maple": "sweet", "sweet italian": "sweet",
    "sweet & sour": "sweet", "sweet & tangy": "sweet", "sweet pepper": "sweet",
    "sweet apple": "sweet", "sweet heat beet": "sweet",

    # === Mushroom ===
    "mushroom": "mushroom", "wild mushroom": "mushroom", "portabello": "mushroom",
    "shiitake mushroom": "mushroom", "mushroom & cheese": "mushroom",
    "mushroom & vegetable": "mushroom", "portabello mushroom & cheese": "mushroom",
    "portabello quinoa": "mushroom", "wild mushroom cauliflower hempseed": "mushroom",
    "savory mushroom": "mushroom", "savory mushroom & roasted garlic": "mushroom",

    # === Veggie ===
    "vegetable": "veggie", "garden vegetable": "veggie", "california vegetable": "veggie",
    "all american vegetable": "veggie", "vegetarian": "veggie", "garden": "veggie",
    "vegetable griller original": "veggie", "vegetable griller prime": "veggie",
    "vegetable chicken": "veggie", "root vegetable": "veggie", "broccoli boost": "veggie",
    "vegetable meat lover": "veggie", "vegetable lovers": "veggie", "garden variety": "veggie",
    "garden broiler": "veggie", "garden herb": "veggie", "garden fresh": "veggie",
    "pumpkin & spinach": "veggie", "falafel & sesame": "veggie", "spinach chicken": "veggie",
    "spinach pesto": "veggie", "asian vegetable": "veggie", "sun dried tomato & spinach": "veggie",
    "herb roasted": "veggie", "eggplant": "veggie", "ultimate black bean": "veggie",
    "sun dried tomato basil": "veggie", "vegan": "veggie", "chick peas & tahini": "veggie",
    "sweet potato & vegetable": "veggie",

    # === Plant Protein / Grain ===
    "black bean": "plant protein", "black rice": "plant protein", "7 grain": "plant protein",
    "5 grain": "plant protein", "multi grain": "plant protein", "lentil": "plant protein",
    "quinoa": "plant protein", "lentil barley": "plant protein", "flax": "plant protein",
    "grain & herb": "plant protein", "grain & seed medley": "plant protein",
    "brown rice & garbanzo & white bean": "plant protein", "beetroot & bean": "plant protein",
    "hazelnut cranberry": "plant protein", "cauliflower": "plant protein",

    # === Herb ===
    "herb & spice": "herb", "garlic & herb": "herb", "savory": "herb", "herby garlic greens": "herb",
    "lemon herb": "herb", "french herb": "herb", "savory tuscan style": "herb",
    "lemon pepper": "herb", "savory orange": "herb", "lemon": "herb", "peppered": "herb",
    "pepper seasoning": "herb", "salt & pepper": "herb", "season & lime": "herb",

    # === Fusion / Other ===
    "italian": "other", "italian style": "other", "chicago italian": "other",
    "italian pepperoni": "other", "bubba": "other", "asian": "other",
    "el capitan": "other", "santa fe": "other", "baja": "other", "sunday funday": "other",
    "el zapatista": "other", "celebration": "other", "ultimate": "other", "the classic": "other",
    "perfect": "other", "feisty": "other", "tasty": "other", "beer": "other",
    "elysian beer": "other", "california burger": "other", "california style": "other",
    "california": "other", "general tsos": "other", "parmigiana": "other",
    "classic pizzeria": "other", "signature stadium dog": "other", "sonoma": "other",
    "the stallion": "other", "new england style": "other", "sunrice": "other",
    "ginger scallion": "other", "chili bean": "other", "salisbury style": "other",
    "roast": "other", "butter": "other", "artichoke": "other",
    "caribbean style plantain": "other", "pepper steak": "other", "pizza pepperoni": "other",
    "walnut & cheese": "other", "mild italian": "other", "bologna": "other"
}
# Convert to lowercase and map using the predefined dictionary
df_selected["Flavor_Grouped"] = df_selected["Flavor / Scent"].str.lower().map(flavor_group_mapping)

# Drop rows where mapping failed (i.e., unknown flavors)
df_selected = df_selected.dropna(subset=["Flavor_Grouped"])

In [ ]:
# Drop original 'Form' and 'Flavor / Scent' columns
# df_selected = df_selected.drop(columns=["Form", "Flavor / Scent"])

# Get unique values from grouped columns
unique_form_groups = df_selected["Form_Grouped"].unique()
unique_flavor_groups = df_selected["Flavor_Grouped"].unique()

# Display results
print("Unique Form Groups:", unique_form_groups)
print("Unique Flavor Groups:", unique_flavor_groups)

In [ ]:
import statsmodels.formula.api as smf


formula = """
   Q('Unit Sales') ~ C(Form_Grouped) + C(Flavor_Grouped) + C(Q('Manufacturer Name'))
                    + Q('Price_Discount_Pct')
                    + Q('ACV Weighted Distribution') +
                    C(Q('Manufacturer Name')):Q('Price_Discount_Pct')

"""
model = smf.ols(formula=formula, data=df_selected).fit()

print(model.summary())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Hardcoded data
form_data = {
    'label': ['ground', 'nugget', 'patty', 'pieces', 'sausage', 'sliced', 'whole'],
    'coef': [122.93, -1549.85, -40.30, -988.16, -1147.78, -1111.95, 155.46],
    'pval': [0.357, 0.000, 0.726, 0.000, 0.000, 0.000, 0.363]
}

flavor_data = {
    'label': ['herb', 'mushroom', 'original', 'other', 'plant protein', 'spicy', 'sweet', 'veggie'],
    'coef': [-243.60, 801.73, 862.09, -386.78, 1000.48, 262.69, -105.27, 963.36],
    'pval': [0.185, 0.000, 0.000, 0.011, 0.000, 0.076, 0.610, 0.000]
}

# Create DataFrames
form_df = pd.DataFrame(form_data)
form_df['significant'] = form_df['pval'] < 0.05

flavor_df = pd.DataFrame(flavor_data)
flavor_df['significant'] = flavor_df['pval'] < 0.05

# Reverse for top-down plotting
form_df = form_df.iloc[::-1].reset_index(drop=True)
flavor_df = flavor_df.iloc[::-1].reset_index(drop=True)

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=100)
fig.suptitle("Effect of Product Form and Flavor on Unit Sales (vs. Base Categories)", fontsize=13, weight='bold')

# --- Product Form Effect ---
axes[0].barh(
    form_df['label'],
    form_df['coef'],
    color=form_df['significant'].map({True: 'blue', False: 'sandybrown'})
)
axes[0].set_title("Product Form Effect", fontsize=11)
axes[0].set_xlabel("Change in Unit Sales vs. Ball", fontsize=10)
axes[0].axvline(0, color='black', linewidth=1, linestyle='dotted')

# --- Flavor Group Effect ---
axes[1].barh(
    flavor_df['label'],
    flavor_df['coef'],
    color=flavor_df['significant'].map({True: 'blue', False: 'sandybrown'})
)
axes[1].set_title("Flavor Group Effect", fontsize=11)
axes[1].set_xlabel("Change in Unit Sales vs. BBQ", fontsize=10)
axes[1].axvline(0, color='black', linewidth=1, linestyle='dotted')

# Legend (manual handle)
from matplotlib.patches import Patch
legend_handles = [
    Patch(color='blue', label='Significant (p < 0.05)'),
    Patch(color='sandybrown', label='Not Significant')
]
axes[0].legend(handles=legend_handles, loc='lower right', fontsize=9)

# Clean layout
for ax in axes:
    ax.tick_params(axis='both', which='major', labelsize=9)
    ax.grid(False)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


In [ ]:
# Define discount range (0% to 30%)
discount_pct = np.linspace(0, 0.30, 100)

# Coefficients from regression output
base_effect = 113500
mfg_alpha_effect = base_effect - 96760
mfg_gamma_effect = base_effect - 104800

# Calculate predicted additional sales
mfg_beta_sales = base_effect * discount_pct
mfg_alpha_sales = mfg_alpha_effect * discount_pct
mfg_gamma_sales = mfg_gamma_effect * discount_pct

# Plotting
plt.figure(figsize=(8, 5), dpi=100)
plt.plot(discount_pct * 100, mfg_beta_sales, label='MFG_BETA (reference)', color='orange')
plt.plot(discount_pct * 100, mfg_alpha_sales, label='MFG_ALPHA', color='red')
plt.plot(discount_pct * 100, mfg_gamma_sales, label='MFG_GAMMA', color='darkred')

plt.title("Impact of Price Discount on Predicted Sales", fontsize=13)
plt.xlabel("Price Discount (%)")
plt.ylabel("Predicted Additional Unit Sales")
plt.legend(frameon=True)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
# Regression coefficient from output
acv_coef = 1376.6422

# ACV range: 10% to 100%
acv_pct = np.arange(10, 101, 10)
acv_values = acv_pct / 100

# Predicted unit sales
predicted_sales = acv_coef * acv_values

# Plot
plt.figure(figsize=(8, 5), dpi=100)
plt.plot(acv_pct, predicted_sales, color='orange', marker='o', linewidth=2)

plt.title("Effect of Availability (ACV) on Unit Sales", fontsize=13)
plt.xlabel("ACV Weighted Distribution (%)")
plt.ylabel("Predicted Unit Sales")
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = model.model.exog
vif = pd.DataFrame()
vif["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
vif["variable"] = model.model.exog_names

print(vif.sort_values("VIF", ascending=False))


## Appendix C — Optional: extract one franchise from large CSV

Chunked read to pull rows matching a franchise string and write `data/franchise_a_filtered.csv`. Skip if you do not need it.


In [ ]:
chunks = pd.read_csv(DATA_PATH, chunksize=100_000)
franchise_a_rows = [
    chunk[chunk["Brand Franchise Name"].str.contains("FRANCHISE_A", na=False, case=False)]
    for chunk in chunks
]
franchise_a_data = pd.concat(franchise_a_rows)
out_path = ROOT / "data" / "franchise_a_filtered.csv"
franchise_a_data.to_csv(out_path, index=False)
print(f"Wrote {len(franchise_a_data):,} rows to {out_path}")
